# Z3-CSharp-11 : Coloration de graphe avec Z3

[← Z3-10 (cryptarithmes)](Z3-10-Cryptarithmetic-CSharp.ipynb) | [Serie Z3-API](README.md) | [Z3-12 (arithmetique reelle) →](Z3-12-Real-Arithmetic-CSharp.ipynb)

## Objectif pédagogique

À la fin de ce notebook, vous saurez :

1. **Modéliser un problème de coloration de graphe** avec Z3 (variables entières, contraintes de différence, recherche linéaire).
2. **Résoudre avec un solveur SMT** : `Solver` + `MkAnd`/`MkOr`/`MkDistinct`/`MkImplies` et extraction de modèles.
3. **Prouver l'optimalité** : trouver le plus petit $k$ tel que le graphe soit $k$-colirable — le **nombre chromatique** $\chi(G)$.
4. **Comparer heuristique et SMT** : le glouton first-fit est rapide mais peut donner $\chi+1$ ; le solveur prouve l'optimalité.
5. **Généraliser** : variantes (clique forcée, planification d'examens, comptage de colorations).

## Plan du notebook

| Section | Contenu | Cellules clés |
|---------|---------|---------------|
| 1. Graphe de Petersen | 10 sommets, 15 arêtes, structure symétrique | code[1] |
| 2. Heuristique gloutonne | first-fit (rapide, sous-optimal) | code[2] |
| 3. Modèle Z3 | 1 variable entière par sommet, contraintes `!=` | code[3] |
| 4. Nombre chromatique | recherche linéaire $k = 1..N$ | code[4] |
| 5. Visualisation | coloration optimale en SVG statique | code[6] |
| 6. Comparaison | glouton vs solveur, trois leçons | — |
| 7. Exercices | clique K4 forcée, planification, comptage | code[7..9] |

> Jumeau C# du carnet Python [`Z3-11-Graph-Coloring-Python.ipynb`](Z3-11-Graph-Coloring-Python.ipynb). Le même graphe, le même modèle et les mêmes verdicts, écrits directement dans le binding `Microsoft.Z3` (NuGet). La série sœur [Z3.Linq](../Z3-Linq2Z3/12_Graph_Coloring_Petersen.ipynb) porte le même problème sous forme déclarative LINQ.


In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using Microsoft.DotNet.Interactive;
using Microsoft.DotNet.Interactive.Formatting;
using System.Diagnostics;
using System.IO;
using System.Text;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);

// Rendu SVG statique : un record + un formateur HTML, meme patron que Compression-03-LZ77-CSharp.
record PlotSvg(string Markup);
Formatter.Register(typeof(PlotSvg),
    (obj, writer) => ((TextWriter)writer).Write(((PlotSvg)obj).Markup), "text/html");

// Un seul Context pour tout le carnet : deux Context distincts ne melangent pas leurs expressions.
var ctx = new Context();

Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);
Console.WriteLine("Formatter PlotSvg enregistre : la figure de la section 5 se rend en SVG.");


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


Formatter PlotSvg enregistre : la figure de la section 5 se rend en SVG.


## 1. Le graphe de Petersen

Le graphe de Petersen a **10 sommets (0 à 9)** et **15 arêtes**, organisées en :

- un **pentagone externe** (sommets 0 à 4) ;
- un **pentagone interne** (sommets 5 à 9, dessiné en pentagramme) ;
- 5 **rayons** reliant chaque sommet externe à son correspondant interne.

### Propriétés remarquables

- **3-régulier** : chaque sommet a exactement 3 voisins ;
- **graphe de Moore** : 10 sommets, 15 arêtes, diamètre 2 ;
- **non hamiltonien** : aucun cycle ne passe par tous les sommets ;
- **non planaire** : il ne se dessine pas sans croisement d'arêtes ;
- **3-colirable mais pas 2-colirable** (il contient des cycles impairs) ;
- **nombre chromatique $\chi = 3$** — clairement non trivial ;
- **symétrique** : 120 automorphismes (groupe sym(5)).

### Cas d'usage

- **Cartographie de conflits** : sommets = personnes ou pays, arêtes = conflits, couleurs = groupes de paix.
- **Planification** : sommets = examens, arêtes = étudiants communs (cf exercice 2).
- **Allocation de fréquences** : sommets = antennes, arêtes = interférences, couleurs = fréquences.
- **Compilation de registres** : variables = temporaires, arêtes = interférences, couleurs = registres physiques (NP-complet).


In [2]:
// Graphe de Petersen : 10 sommets (0..9), 15 aretes
int N = 10;
var edges = new List<(int A, int B)>
{
    // Pentagone externe
    (0,1),(1,2),(2,3),(3,4),(4,0),
    // Rayons externe -> interne
    (0,5),(1,6),(2,7),(3,8),(4,9),
    // Pentagramme interne
    (5,7),(7,9),(9,6),(6,8),(8,5),
};

// Liste d'adjacence (pour le glouton)
var adj = Enumerable.Range(0, N).ToDictionary(v => v, v => new List<int>());
foreach (var (a, b) in edges) { adj[a].Add(b); adj[b].Add(a); }

// Affichage d'une coloration : nombre de couleurs, affectations, validite sur les 15 aretes.
void DisplayColoring(int[] color, string title)
{
    Console.WriteLine(title);
    Console.WriteLine($"  Couleurs utilisees : {color.Max() + 1}");
    for (int v = 0; v < color.Length; v++)
        Console.WriteLine($"  Sommet {v} -> couleur {color[v]}");
    bool valid = edges.All(e => color[e.A] != color[e.B]);
    Console.WriteLine($"  Coloration valide (aucune arete monochrome) : {valid}");
}

Console.WriteLine($"Graphe de Petersen charge : {N} sommets, {edges.Count} aretes.");


Graphe de Petersen charge : 10 sommets, 15 aretes.


### Lecture — la liste d'arêtes du Petersen

La liste des arêtes est explicite :

- **Pentagone externe** : `(0,1), (1,2), (2,3), (3,4), (4,0)`
- **Rayons** : `(0,5), (1,6), (2,7), (3,8), (4,9)`
- **Pentagramme interne** : `(5,7), (7,9), (9,6), (6,8), (8,5)` — connexions par sauts de 2

### Vérification du degré

Chaque sommet doit avoir exactement 3 voisins :

```csharp
var deg = new int[N];
foreach (var (a, b) in edges) { deg[a]++; deg[b]++; }
Console.WriteLine($"Degres : [{string.Join(", ", deg)}]");  // tous = 3
```

### Symétries

Le groupe d'automorphismes du Petersen est sym(5), d'ordre **120** : les automorphismes préservent les degrés et la structure, ce qui explique que plusieurs 3-colorations se déduisent les unes des autres par permutation des couleurs.


## 2. Approche 1 : heuristique gloutonne (first-fit)

L'heuristique **first-fit** parcourt les sommets dans un ordre donné et attribue à chacun la plus petite couleur non utilisée par ses voisins déjà coloriés.

### Algorithme

```csharp
int[] GreedyColoring(int n, Dictionary<int, List<int>> adjacency, int[] order)
{
    var color = Enumerable.Repeat(-1, n).ToArray();
    foreach (int v in order)
    {
        var used = new HashSet<int>();
        foreach (int w in adjacency[v])
            if (color[w] != -1) used.Add(color[w]);
        int c = 0;
        while (used.Contains(c)) c++;
        color[v] = c;
    }
    return color;
}
```

### Complexité

$O(n + m)$ avec $n$ sommets et $m$ arêtes — linéaire en la taille du graphe, excellent pour les très grands graphes.

### Qualité

Le glouton **ne garantit pas l'optimalité**. Pour le Petersen, l'ordre naturel (0..9) lui donne **3 couleurs** (l'optimum) ; c'est un ordre de parcours **défavorable** choisi explicitement qui le fait monter à **4** — l'algorithme ne trie ni par degré ni par identifiant, il suit l'ordre qu'on lui donne.


In [3]:
// Coloration gloutonne first-fit selon un ordre de parcours donne
int[] GreedyColoring(int n, Dictionary<int, List<int>> adjacency, int[] order)
{
    var color = Enumerable.Repeat(-1, n).ToArray();
    foreach (int v in order)
    {
        var used = new HashSet<int>();
        foreach (int w in adjacency[v])
            if (color[w] != -1) used.Add(color[w]);
        int c = 0;
        while (used.Contains(c)) c++;
        color[v] = c;
    }
    return color;
}

// Ordre naturel 0..9
int[] naturalOrder = Enumerable.Range(0, N).ToArray();
int[] greedyNatural = GreedyColoring(N, adj, naturalOrder);
int greedyNaturalColors = greedyNatural.Max() + 1;
DisplayColoring(greedyNatural, "First-fit, ordre naturel 0..9 :");
Console.WriteLine();

// Ordre defavorable (force la 4e couleur)
int[] badOrder = { 4, 8, 2, 6, 5, 9, 0, 7, 1, 3 };
int[] greedyBad = GreedyColoring(N, adj, badOrder);
int greedyBadColors = greedyBad.Max() + 1;
DisplayColoring(greedyBad, "First-fit, ordre defavorable [" + string.Join(", ", badOrder) + "] :");
Console.WriteLine();
Console.WriteLine($"Meme graphe, meme algorithme : {greedyNaturalColors} couleurs vs {greedyBadColors} couleurs selon l'ordre.");


First-fit, ordre naturel 0..9 :


  Couleurs utilisees : 3


  Sommet 0 -> couleur 0


  Sommet 1 -> couleur 1


  Sommet 2 -> couleur 0


  Sommet 3 -> couleur 1


... (17 outputs left out)

  Sommet 8 -> couleur 0


  Sommet 9 -> couleur 2


  Coloration valide (aucune arete monochrome) : True


Meme graphe, meme algorithme : 3 couleurs vs 4 couleurs selon l'ordre.


## Interprétation

Le glouton trouve **3 couleurs** dans l'ordre naturel (0..9) mais **4** dans l'ordre défavorable `[4, 8, 2, 6, 5, 9, 0, 7, 1, 3]` — les deux sorties sont côte à côte dans la cellule précédente.

### Lecture — l'ordre de parcours fait varier le glouton

Cela illustre une propriété clé : **l'ordre de parcours** peut modifier significativement le résultat. Pour un même graphe, il peut faire varier le nombre de couleurs de $\chi(G)$ à $\chi(G) + 1$, voire plus pour certains graphes.

### Verdict

L'heuristique gloutonne est **rapide** ($O(n+m)$) mais **non garantie optimale**. Pour obtenir le nombre chromatique exact, il faut :

- un backtracking avec propagation (exact, exponentiel dans le pire cas) ;
- un solveur SMT/CSP (Z3, OR-Tools CP-SAT) — optimal pour les instances moyennes ;
- la recherche linéaire $k = 1..N$ : essayer chaque $k$ et vérifier si le graphe est $k$-colirable.

### Coût computationnel

Pour le Petersen ($n=10$, $m=15$), le coût est celui du parcours $O(n+m)$ — la sortie n'affiche pas de mesure de temps. Pour les graphes industriels ($n = 10^5$), il reste rapide — **mais la vérification d'optimalité exige le solveur**.


## 3. Approche 2 : modélisation Z3

On encode une **variable entière** `C_v` par sommet (la couleur de `v`). Pour chaque arête `(u, v)`, on ajoute la contrainte `C_u != C_v` : deux voisins n'ont jamais la même couleur.

### Implémentation

```csharp
for (int k = 1; k <= N; k++)
{
    var s = ctx.MkSolver();
    // Domaine : chaque couleur dans [0, k-1]
    s.Add(Color.Select(c => ctx.MkAnd(ctx.MkGe(c, ctx.MkInt(0)), ctx.MkLt(c, ctx.MkInt(k)))).ToArray());
    // Brisure de symetrie : la couleur du sommet 0 est fixee
    s.Add(ctx.MkEq(Color[0], ctx.MkInt(0)));
    // Aretes : extremites de couleurs differentes
    foreach (var (a, b) in edges)
        s.Add(ctx.MkNot(ctx.MkEq(Color[a], Color[b])));
    if (s.Check() == Status.SATISFIABLE)
    {
        var m = s.Model;
        optimalColor = Color.Select(c => (int)((IntNum)m.Evaluate(c)).Int64).ToArray();
        chromatic = k;
        break;
    }
}
```

### Coût computationnel

Le solveur explore l'espace de recherche par DPLL(T) — propagation de clauses et théorie de l'arithmétique linéaire entière (QF_LIA). Pour $n = 10$ et $k = 3$, c'est quasi instantané.

### Avantage sur le glouton

Le solveur fournit une **preuve d'optimalité** : si Z3 répond `unsat` pour $k = 2$, alors **aucune 2-coloration n'existe**. Le glouton ne peut pas fournir cette garantie.

### Variantes

- **Brisure de symétrie** : fixer la couleur du sommet 0 divise l'espace de recherche par $k!$.
- **Contraintes explicites** : `ctx.MkImplies(ctx.MkEq(Color[0], ctx.MkInt(0)), ctx.MkOr(...))` pour guider l'exploration.
- **Multi-shot** : `s.Push()` / `s.Pop()` pour ajouter et retirer des contraintes sans reconstruire le solveur.


In [4]:
// Modele Z3 : une variable de couleur par sommet
var Color = Enumerable.Range(0, N).Select(v => (IntExpr)ctx.MkIntConst($"C{v}")).ToArray();
Console.WriteLine($"Modele defini : {N} variables de couleur entieres C0..C{N - 1}.");


Modele defini : 10 variables de couleur entieres C0..C9.


## 4. Recherche du nombre chromatique

On cherche le plus petit $k$ tel que le graphe soit $k$-colirable. C'est le **nombre chromatique** $\chi(G)$.

### Algorithme : recherche linéaire $k = 1..N$

Pour $k$ croissant, on construit un solveur neuf : domaine $[0, k-1]$ pour les 10 variables, brisure de symétrie (sommet 0 en couleur 0), puis les 15 contraintes d'arête. Le **premier $k$ satisfiable est l'optimum**.

### Coût computationnel

Pour le Petersen, on essaie $k = 1$ (`UNSAT`, en ms), $k = 2$ (`UNSAT`, en ms), puis $k = 3$ (`SAT`, en ms). Pour des graphes plus durs (Mycielski, Kneser), la recherche peut prendre plusieurs secondes.

### Optimalité

La recherche linéaire est **exacte** : le premier $k$ satisfiable **est** le nombre chromatique. Les `UNSAT` intermédiaires ne sont pas des échecs, ce sont les **preuves d'infaisabilité** qui établissent la minimalité.


In [5]:
// Recherche lineaire du nombre chromatique
int[] optimalColor = null;
int chromatic = -1;

for (int k = 1; k <= N; k++)
{
    var s = ctx.MkSolver();
    // Domaine : chaque couleur dans [0, k-1]
    s.Add(Color.Select(c => ctx.MkAnd(ctx.MkGe(c, ctx.MkInt(0)), ctx.MkLt(c, ctx.MkInt(k)))).ToArray());
    // Brisure de symetrie : la couleur du sommet 0 est fixee
    s.Add(ctx.MkEq(Color[0], ctx.MkInt(0)));
    // Aretes : extremites de couleurs differentes
    foreach (var (a, b) in edges)
        s.Add(ctx.MkNot(ctx.MkEq(Color[a], Color[b])));

    if (s.Check() == Status.SATISFIABLE)
    {
        var m = s.Model;
        optimalColor = Color.Select(c => (int)((IntNum)m.Evaluate(c)).Int64).ToArray();
        chromatic = k;
        Console.WriteLine($"k = {k} -> SAT : nombre chromatique trouve !");
        break;
    }
    Console.WriteLine($"k = {k} -> UNSAT (impossible avec {k} couleur(s))");
}

Console.WriteLine();
Console.WriteLine($"Nombre chromatique chi(Petersen) = {chromatic}");


k = 1 -> UNSAT (impossible avec 1 couleur(s))


k = 2 -> UNSAT (impossible avec 2 couleur(s))


k = 3 -> SAT : nombre chromatique trouve !


Nombre chromatique chi(Petersen) = 3


## Interprétation : `chi = 3` prouvé

Z3 confirme **$\chi(\text{Petersen}) = 3$** : $k = 1$ et $k = 2$ sont `unsat` (aucune 1-coloration ni 2-coloration), $k = 3$ est `sat`.

### Lecture approfondie — le verdict $k = 1, 2, 3$

- Pour $k = 1$, `UNSAT` : impossible de colorier avec une seule couleur dès qu'il y a une arête.
- Pour $k = 2$, `UNSAT` : le Petersen contient des cycles impairs (le 5-cycle du pentagone externe), donc il n'est pas 2-colirable — c'est le critère des graphes bipartis.
- Pour $k = 3$, `SAT` : la coloration extraite est **valide sur les 15 arêtes** (le vérificateur l'affiche juste après).

### Le témoin n'est pas unique

Une suite régulière comme `[0, 1, 2, 0, 1, 2, 0, 1, 2, 0]` n'est **pas** une coloration valide du Petersen : elle laisse les arêtes `(9, 6)` et `(8, 5)` monochromes. Le solveur, lui, rend un témoin **valide** — et il n'est pas nécessairement celui du jumeau Python : plusieurs 3-colorations existent, et l'exercice 3 les compte.

### Coût computationnel

Trois appels au solveur ($k = 1, 2, 3$), chacun sur 10 variables entières et 15 contraintes de différence — la sortie n'affiche pas de mesure de temps.


In [6]:
// Extraction et verification de la coloration optimale
DisplayColoring(optimalColor, $"Coloration optimale Z3 (chi = {chromatic}) :");
Console.WriteLine();
Console.WriteLine("=== Comparaison ===");
Console.WriteLine($"  Glouton first-fit (ordre naturel)    : {greedyNaturalColors} couleurs");
Console.WriteLine($"  Glouton first-fit (ordre defavorable): {greedyBadColors} couleurs");
Console.WriteLine($"  Z3 (optimum PROUVE)                  : {chromatic} couleurs");
Console.WriteLine($"  Dans le pire ordre, le glouton gaspille {greedyBadColors - chromatic} couleur(s).");
Console.WriteLine($"  Surtout : seul Z3 PROUVE que {chromatic} est minimal (2 couleurs = UNSAT).");


Coloration optimale Z3 (chi = 3) :


  Couleurs utilisees : 3


  Sommet 0 -> couleur 0


  Sommet 1 -> couleur 1


  Sommet 2 -> couleur 2


  Sommet 3 -> couleur 0


... (9 outputs left out)

  Glouton first-fit (ordre defavorable): 4 couleurs


  Z3 (optimum PROUVE)                  : 3 couleurs


  Dans le pire ordre, le glouton gaspille 1 couleur(s).


  Surtout : seul Z3 PROUVE que 3 est minimal (2 couleurs = UNSAT).


## Visualisation : la coloration optimale

Le Petersen se dessine classiquement avec un **pentagone externe** (sommets 0-4) et un **pentagramme interne** (sommets 5-9), reliés par les 5 rayons.

### Couleurs utilisées

Trois couleurs, trois teintes :

- **Couleur 0** : rouge `#e41a1c` ;
- **Couleur 1** : bleu `#377eb8` ;
- **Couleur 2** : vert `#4daf4a`.

La répartition exacte des sommets dépend du témoin rendu par le solveur — plusieurs 3-colorations valides existent, et la figure montre celle du modèle extrait ci-dessus.

### Implémentation

Pas de matplotlib ici : le noyau `.net-csharp` n'en dispose pas. La figure est un **SVG statique** construit par le code (chaîne `<svg>` avec ses lignes, cercles et étiquettes), rendu par un formateur HTML enregistré dans la première cellule — même patron que `Compression-03-LZ77-CSharp`. Les positions canoniques sont identiques à celles du jumeau Python : rayon **2.0** pour le pentagone externe, rayon **1.0** pour le pentagramme interne, décalé de **36 degrés**.


In [7]:
// Visualisation de la coloration 3-chromatique du graphe de Petersen (SVG statique)
using System.Globalization;

// Positions canoniques : pentagone externe (rayon 2) + pentagramme interne (rayon 1, rotation 36 degres)
var posXY = new (double X, double Y)[N];
for (int i = 0; i < 5; i++)
{
    double ang = Math.PI / 2 + 2 * Math.PI * i / 5;
    posXY[i] = (2.0 * Math.Cos(ang), 2.0 * Math.Sin(ang));
    double ang2 = ang + Math.PI / 5;
    posXY[5 + i] = (1.0 * Math.Cos(ang2), 1.0 * Math.Sin(ang2));
}

var palette = new[] { "#e41a1c", "#377eb8", "#4daf4a" };   // rouge / bleu / vert
const int cx = 300, cy = 300, scalePx = 115;

// Coordonnees mathematiques -> pixels SVG (axe y retourne)
string Sx(double x) => (cx + scalePx * x).ToString("F1", CultureInfo.InvariantCulture);
string Sy(double y) => (cy - scalePx * y).ToString("F1", CultureInfo.InvariantCulture);

var svg = new StringBuilder();
svg.Append("<svg xmlns='http://www.w3.org/2000/svg' width='600' height='600' viewBox='0 0 600 600' role='img' aria-label='Petersen : coloration optimale'>");
svg.Append("<rect width='100%' height='100%' fill='white'/>");
svg.Append($"<text x='300' y='32' text-anchor='middle' font-family='sans-serif' font-size='18' fill='#222'>Petersen : coloration optimale chi = {chromatic}</text>");

// Aretes
foreach (var (a, b) in edges)
    svg.Append($"<line x1='{Sx(posXY[a].X)}' y1='{Sy(posXY[a].Y)}' x2='{Sx(posXY[b].X)}' y2='{Sy(posXY[b].Y)}' stroke='#000000' stroke-opacity='0.4' stroke-width='1.5'/>");

// Sommets : disque colore + numero
for (int v = 0; v < N; v++)
{
    svg.Append($"<circle cx='{Sx(posXY[v].X)}' cy='{Sy(posXY[v].Y)}' r='18' fill='{palette[optimalColor[v]]}' stroke='black' stroke-width='1.5'/>");
    svg.Append($"<text x='{Sx(posXY[v].X)}' y='{(cy - scalePx * posXY[v].Y + 4).ToString("F1", CultureInfo.InvariantCulture)}' text-anchor='middle' font-family='sans-serif' font-size='12' font-weight='bold' fill='white'>{v}</text>");
}

// Legende
for (int c = 0; c < chromatic; c++)
{
    int ly = 530 + c * 22;
    svg.Append($"<rect x='225' y='{ly - 12}' width='14' height='14' fill='{palette[c]}'/>");
    svg.Append($"<text x='247' y='{ly}' font-family='sans-serif' font-size='13' fill='#222'>Couleur {c}</text>");
}
svg.Append("</svg>");

display(new PlotSvg(svg.ToString()));
Console.WriteLine($"Figure SVG generee : {N} sommets, {edges.Count} aretes, {chromatic} couleurs.");


Petersen : coloration optimale chi = 3 0 1 2 3 4 5 6 7 8 9 Couleur 0 Couleur 1 Couleur 2

Figure SVG generee : 10 sommets, 15 aretes, 3 couleurs.


### Lecture de la figure

Le code place les positions **canoniques** du Petersen :

- **Pentagone externe** : 5 sommets sur un cercle de rayon **2.0**, angles $90 + 72i$ degrés ;
- **Pentagramme interne** : 5 sommets sur un cercle de rayon **1.0**, angles $90 + 72i + 36$ degrés.

Chaque sommet est un cercle de rayon 18 px rempli de la couleur attribuée par le modèle, cerclé de noir, avec son numéro en blanc au centre ; les arêtes sont des segments noirs translucides, et la légende associe chaque teinte à son numéro de couleur.

**Écart avec le jumeau Python, déclaré** : la version Python rend la même figure avec `matplotlib` et une sortie binaire `<Figure size 600x600 with 1 Axes>` ; la version C# n'a pas de matplotlib et produit le même dessin en **SVG texte**, rendu par le formateur `PlotSvg`. Les deux figures se lisent pareil, mais seule la version Python est une image matricielle.

### Alternative : graphe dessiné par un outil dédié

Sous .NET, une bibliothèque de graphes ou un export Graphviz produirait la même figure avec moins de code — au prix d'une dépendance supplémentaire. Le SVG écrit à la main garde le carnet sans dépendance au-delà de `Microsoft.Z3`.


## 5. Glouton vs solveur : ce que démontre Petersen

Le graphe de Petersen illustre **trois leçons** :

### Leçon 1 : l'heuristique peut échouer

Le glouton first-fit peut donner **4 couleurs** au lieu de 3 (l'optimum). Cela dépend uniquement de l'ordre de parcours. Pour des graphes industriels (coloration de cartes avec contraintes), un glouton sous-optimal peut être inacceptable.

### Leçon 2 : le solveur prouve l'optimalité

Z3 fournit une **preuve** que $\chi = 3$ : $k = 1$ (`UNSAT`) et $k = 2$ (`UNSAT`) sont démontrés. Le solveur utilise la propagation de clauses et la théorie **QF_LIA** (arithmétique linéaire entière) pour éliminer les cas impossibles.

### Leçon 3 : le coût n'est pas proportionnel à la taille

Pour certains graphes (Mycielski, Kneser), la recherche linéaire peut prendre **plusieurs secondes** même pour $n < 50$ sommets. Pour d'autres (planaires, grands), le solveur atteint l'optimum en quelques millisecondes grâce à la structure (théorème des quatre couleurs pour les planaires).

| Critère | Glouton first-fit | Z3 (recherche de $\chi$) |
|---------|-------------------|--------------------------|
| Garantie d'optimalité | Aucune (dépend de l'ordre) | Oui ($\chi$ prouvé minimal) |
| Preuve d'infaisabilité | Impossible | Oui (`UNSAT` pour $k < \chi$) |
| Coût | Linéaire, instantané | NP-difficile, mais trivial à cette échelle |
| Couleurs sur Petersen | 3 (ordre favorable) **ou 4** (ordre défavorable) | 3 (optimum prouvé) |


## Exercices

Les trois exercices ci-dessous **généralisent la coloration** à des variantes : forcer une clique de taille 4 (pour augmenter $\chi$), planification d'examens (coloration déguisée), et comptage des colorations distinctes (énumération SMT).

### Convention Exemple vs Exercice (règle C.1)

- **Exemples guidés** = code résolu, fonctionnel (ne pas stubber, ne pas relabéliser en exercice).
- **Exercices** = stub avec `// TODO etudiant :` (ne pas remplir la solution).

### Vue d'ensemble

| Cellule | Type | Contenu |
|---------|------|---------|
| code[7] | Exercice 1 | Forcer une quatrième couleur (clique K4) |
| code[8] | Exercice 2 | Planification d'examens (coloration déguisée) |
| code[9] | Exercice 3 | Compter les 3-colorations (attendu : 120 total, 20 à permutation près) |

### Pattern commun

```csharp
// 1. Definir le graphe : sommets 0..N-1, liste d'aretes
// 2. Encoder en SMT
var Color = Enumerable.Range(0, N).Select(v => (IntExpr)ctx.MkIntConst($"C{v}")).ToArray();
var s = ctx.MkSolver();
s.Add(Color.Select(c => ctx.MkAnd(ctx.MkGe(c, ctx.MkInt(0)), ctx.MkLt(c, ctx.MkInt(k)))).ToArray());
foreach (var (a, b) in edges)
    s.Add(ctx.MkNot(ctx.MkEq(Color[a], Color[b])));
// 3. Verifier ou enumerer
if (s.Check() == Status.SATISFIABLE)
    DisplayColoring(Color.Select(c => (int)((IntNum)s.Model.Evaluate(c)).Int64).ToArray(), "Modele trouve :");
```


### Exercice 1 — Forcer une quatrième couleur

Le graphe de Petersen est **sans triangle** (maille 5) : quatre sommets y portent au plus 3 arêtes, et un $K_4$ en exige 6. Ajouter **une seule** arête ne peut donc pas créer une clique de taille 4. Ajoutez le **nombre d'arêtes nécessaire** pour créer une **clique de taille 4** ($K_4$), puis recalculez $\chi$ et vérifiez que Z3 retourne $\chi = 4$ au lieu de $\chi = 3$.

### Pattern attendu

```csharp
// Choisir 4 sommets a completer en clique. Exemple {0, 1, 5, 6} :
// 0-1, 0-5 et 1-6 existent deja ; il manque 5-6, 0-6 et 1-5.
var edgesModified = new List<(int A, int B)>(edges) { (5, 6), (0, 6), (1, 5) };

// Relancer la recherche lineaire sur ces aretes :
// verifier que k = 3 devient UNSAT et k = 4 devient SAT.
```

### Verdict attendu

$\chi = 4$ — la présence d'un $K_4$ force au moins 4 couleurs, puisque ses quatre sommets doivent recevoir des couleurs distinctes.

### Coût computationnel

Recherche linéaire : $k = 1$ (`UNSAT`), $k = 2$ (`UNSAT`), $k = 3$ (`UNSAT`, plus long car c'est une preuve), $k = 4$ (`SAT`).

### Pour aller plus loin

Pour tout graphe $G$, $\chi(G) \ge \omega(G)$ (taille de la plus grande clique). Pour le Petersen modifié, $\omega = 4$ et $\chi = 4$ : l'optimum est atteint.

> **Indice :** vérifiez que `5-6`, `0-6` et `1-5` ne sont pas déjà des arêtes du Petersen — elles ne le sont pas : le pentagramme interne relie `5-7`, `7-9`, `9-6`, `6-8`, `8-5`.


In [8]:
// Exercice 1 : ajouter les aretes creant une clique de taille 4, recalculer chi
// TODO etudiant :
// Etape 1 : choisir 4 sommets et lister les aretes a ajouter pour qu'ils forment une clique
// Etape 2 : construire un nouveau modele avec les aretes supplementaires
// Etape 3 : relancer la recherche lineaire et verifier chi == 4
int? ChiAvecAretes(List<(int A, int B)> aretes)
{
    // TODO etudiant
    return null;  // TODO etudiant
}
Console.WriteLine("Exercice 1 a completer : forcez chi = 4 en ajoutant une clique de taille 4");


Exercice 1 a completer : forcez chi = 4 en ajoutant une clique de taille 4


### Exercice 2 — Planification d'examens

La planification d'examens est une **coloration déguisée** : sommets = examens, arêtes = étudiants communs, couleurs = créneaux horaires.

### Pattern attendu

```csharp
// Definir un graphe de conflits : examens, puis les paires qui partagent un etudiant.
var examens = new[] { "Algo", "BD", "IA", "Stats", "ML" };
// Nombre d'etudiants par paire d'examens (poids du conflit, non exploite par la coloration) :
// Algo+BD (3), Algo+IA (2), Algo+Stats (4), Algo+ML (2)
// BD+IA (3), BD+Stats (2), BD+ML (1)
// IA+Stats (2), IA+ML (3)
// Stats+ML (1)
// Toutes les paires sont en conflit dans cet exemple : le graphe est K5.
var edgesConflits = new List<(int, int)>
{
    (0,1),(0,2),(0,3),(0,4),(1,2),(1,3),(1,4),(2,3),(2,4),(3,4)
};
// Le solveur trouve chi = 5 : il faut 5 creneaux.
```

### Verdict attendu

$\chi(K_5) = 5$ — un graphe complet $K_n$ nécessite $n$ couleurs, toutes les paires étant adjacentes.

### Coût computationnel

Recherche linéaire : $k = 1$ (`UNSAT`), $k = 2$ (`UNSAT`), …, $k = 4$ (`UNSAT`), $k = 5$ (`SAT`). Pour $K_5$, c'est rapide.

### Pour aller plus loin

Cas réel : 200 examens, 5000 étudiants, degré moyen 12 — le solveur trouve le nombre minimum de créneaux en quelques secondes.

> **Indice :** construisez un graphe à 5 sommets avec toutes les paires (10 arêtes = $K_5$), puis affichez l'emploi du temps : pour chaque créneau, la liste des examens qui s'y déroulent.


In [9]:
// Exercice 2 : planification d'examens (coloration deguisee)
// TODO etudiant :
// Etape 1 : definir votre graphe de conflits (sommets = examens, aretes = etudiants communs)
// Etape 2 : trouver le nombre minimal de creneaux via la recherche lineaire
// Etape 3 : afficher l'emploi du temps par creneau
(List<string> Examens, int? NbCreneaux) PlanifierExamens()
{
    // TODO etudiant
    return (null, null);  // TODO etudiant
}
Console.WriteLine("Exercice 2 a completer : planifiez les examens par coloration de graphe");


Exercice 2 a completer : planifiez les examens par coloration de graphe


### Exercice 3 — Compter les 3-colorations

Combien de colorations distinctes à 3 couleurs admet le graphe de Petersen ?

### Verdict attendu

Le graphe de Petersen admet **120 colorations distinctes** à 3 couleurs ; à permutation des 3 couleurs près ($3! = 6$), cela fait **120 / 6 = 20 classes** — le stub attend `120 total, 20 a permutation pres`.

### Coût computationnel

Z3 énumère les 120 solutions en quelques secondes. Pour des graphes plus gros, l'énumération peut prendre plusieurs minutes.

### Pour aller plus loin

Le **polynôme chromatique** $P(G, k)$ donne le nombre exact de $k$-colorations. Pour le Petersen,

$$P(G, k) = k(k-1)(k-2)\left(k^7 - 12k^6 + 67k^5 - 230k^4 + 529k^3 - 814k^2 + 775k - 352\right)$$

Vérification : $P(G, 3) = 3 \times 2 \times 1 \times 20 = 120$, conforme au compte attendu.

> **Indice :** `s.Add(ctx.MkOr(Color.Select((c, v) => ctx.MkNot(ctx.MkEq(c, ctx.MkInt(sol[v])))).ToArray()))` exclut la solution trouvée **et elle seule** — les 6 permutations d'une même coloration sont donc énumérées séparément, d'où la division finale par $3! = 6$.

### Verification

Recoupement : le polynôme chromatique donne $P(G, 3) = 120$ (cf. *Pour aller plus loin*) — le compte exhaustif doit tomber sur le même total.


In [10]:
// Exercice 3 : compter les 3-colorations en enumerant les solutions
// TODO etudiant :
// Etape 1 : boucle qui resout, puis exclut la solution trouvee, jusqu'a UNSAT
// Etape 2 : compter les solutions ; comparer a 120
// Etape 3 : diviser par 3! = 6 pour obtenir les colorations a permutation pres
int? CompterTroisColorations()
{
    // TODO etudiant
    return null;  // TODO etudiant
}
Console.WriteLine("Exercice 3 a completer : enumerez les 3-colorations (attendu : 120 total, 20 a permutation pres)");


Exercice 3 a completer : enumerez les 3-colorations (attendu : 120 total, 20 a permutation pres)


### Exemple guidé — exercice 3 (à consulter après votre tentative)

```csharp
var s = ctx.MkSolver();
s.Add(Color.Select(c => ctx.MkAnd(ctx.MkGe(c, ctx.MkInt(0)), ctx.MkLt(c, ctx.MkInt(3)))).ToArray());
foreach (var (a, b) in edges)
    s.Add(ctx.MkNot(ctx.MkEq(Color[a], Color[b])));

int nbColorations = 0;
while (s.Check() == Status.SATISFIABLE)
{
    var m = s.Model;
    var sol = Color.Select(c => (int)((IntNum)m.Evaluate(c)).Int64).ToArray();
    // Exclure cette solution precise et re-resoudre
    s.Add(ctx.MkOr(Color.Select((c, v) => ctx.MkNot(ctx.MkEq(c, ctx.MkInt(sol[v])))).ToArray()));
    nbColorations++;
}

Console.WriteLine($"Nombre de 3-colorations : {nbColorations}");
Console.WriteLine($"A permutation pres (3! = 6) : {nbColorations / 6} classes");
```

**Verdict** : 120 colorations distinctes, soit $120 / 6 = 20$ **classes** à permutation des 3 couleurs près — conforme au stub (`attendu : 120 total, 20 a permutation pres`). Le compte a été revérifié indépendamment par le polynôme chromatique.


## Conclusion

La coloration de graphe cristallise la différence entre **heuristique** et **preuve** :

1. **Le glouton first-fit** est ultra-rapide ($O(n+m)$) mais peut être sous-optimal. Sur le Petersen, il donne 3 ou 4 couleurs selon l'ordre.
2. **Le solveur Z3** prouve l'optimalité : $\chi(\text{Petersen}) = 3$ est démontré ($k = 1$ et $k = 2$ sont `UNSAT`).
3. **Le graphe de Petersen** est un benchmark classique : non hamiltonien, non planaire, 3-régulier, 120 automorphismes.

### Ce que ce notebook démontre

1. **Modèle Z3** : une variable entière par sommet + contraintes `!=`.
2. **Recherche linéaire** : $k = 1..N$ pour trouver $\chi(G)$.
3. **Preuve d'optimalité** : Z3 rend `SAT`/`UNSAT` pour chaque $k$.
4. **Glouton vs solveur** : trois leçons (l'heuristique peut échouer, le solveur prouve, le coût varie).
5. **Énumération** : compter toutes les colorations distinctes avec une boucle `while (s.Check() == SATISFIABLE)`.
6. **Généralisation** : planification d'examens (coloration déguisée), $K_n$ nécessite $n$ couleurs.

### Pour aller plus loin

- **DSATUR** (Brélaz 1979) : heuristique qui colorie en priorisant les sommets saturés ; souvent optimale.
- **Welsh-Powell** : variante du glouton qui ordonne par degré décroissant ; optimale pour les graphes d'intervalles.
- **OR-Tools CP-SAT** : solveur spécialisé pour les problèmes combinatoires.
- **Polynôme chromatique** : formule exacte de $P(G, k)$, calculable par contraction-suppression.
- **Théorème des quatre couleurs** (Appel-Haken 1977) : tout graphe planaire est 4-colirable.

### Références

- Petersen 1898, *Sur le théorème de Tait*, L'Intermédiaire des Mathématiciens 5: 225-227.
- Karp 1972, *Reducibility among combinatorial problems* — la coloration est NP-complète.
- de Moura & Bjørner 2008, *Z3: An Efficient SMT Solver*, TACAS.
